# 00 · Environment and evidence

**Question:** Which data, code and environment produced the results?

The five public notebooks are executed views of checksummed competition-data aggregates. Private comments, labels by row, model weights and prediction files are excluded. Opening or executing these notebooks needs no AWS account or model downloads.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from jigsaw_rules.review import public_evidence
from jigsaw_rules.runtime import environment

root = Path(os.environ.get("JIGSAW_ROOT", Path.cwd())).resolve()
if root.name == "notebooks":
    root = root.parent
baseline = public_evidence(root, "baseline")
semantic = public_evidence(root, "semantic")
assert baseline["training_sha256"] == semantic["training_sha256"]
print("Completed feature experiments | 2,029 rows | two labeled rules")
print("Aggregate checksums verified. This notebook performs no model fitting.")
protocols = {"seen_rule": "Familiar rules", "heldout_rule": "Held-out rule"}

def metric_table(records, heldout=False):
    return pd.DataFrame([{"Representation": r["model"], "Validation": protocols[r["protocol"]],
        "Rule macro AUC": r["metrics"]["rule_macro_auc"],
        "Log loss": r["metrics"]["log_loss"], "Brier": r["metrics"]["brier"],
        "Average precision": r["metrics"]["average_precision"]}
        for r in records if not heldout or r["protocol"] == "heldout_rule"]).round(4)


Completed feature experiments | 2,029 rows | two labeled rules
Aggregate checksums verified. This notebook performs no model fitting.


## Reproduction environment
The original reference environment and the current notebook environment are separate records. `uv.lock` fixes dependencies; a historical result is not relabeled as a fresh experiment.

In [2]:
recorded = baseline["provenance"]["environment"]
current = environment()
print("Recorded Python:", recorded["python"])
print("Notebook Python:", current["python"])
display(pd.DataFrame({"Original reference": recorded["packages"], "Notebook execution": current["packages"]}))

Recorded Python: 3.12.14
Notebook Python: 3.12.13


,Original reference,Notebook execution
numpy,2.3.5,2.3.5
pandas,2.2.3,2.2.3
scipy,1.17.0,1.17.0
scikit-learn,1.8.0,1.8.0
joblib,1.5.3,1.5.3
plotly,7.0.0,7.0.0


## Data identity and limits
Each observation contains a comment, rule, community, four labeled support examples and a binary target. There are no timestamps or longitudinal entity records. The ten-row preview test is not independent evidence of performance.

In [3]:
audit = baseline["audit"]
display(pd.DataFrame({"File": ["train.csv", "test.csv (preview)"], "Rows": [audit["train_rows"], audit["preview_test_rows"]]}))
print("Training SHA-256:", baseline["training_sha256"])
print("Reference run:", baseline["run_id"])

,File,Rows
0,train.csv,2029
1,test.csv (preview),10


Training SHA-256: 83948d06a1e4b16421b738add60ef489cf1d44a2349ca711958fbb41c6207a0a
Reference run: c15c2c2318fc0ed619c6


## Research lineage
Each study records its source fingerprint, input identity, configuration and complete-stage checksums. The notebook executor additionally binds its cache to source, reports, environment and executed output.

In [4]:
import sys
sys.path.insert(0, str(root / "scripts"))
from build_research_report import display_figure
from build_release_report import display_boundary
from jigsaw_rules.features import feature_evidence
from jigsaw_rules.research import research_evidence
from jigsaw_rules.diagnostics import diagnostic_evidence
from jigsaw_rules.pairs import pairs_evidence
from jigsaw_rules.robustness import robustness_evidence
from jigsaw_rules.gate import feature_gate
from jigsaw_rules.instructions import instruction_evidence
from jigsaw_rules.released import released_evidence

controls = feature_evidence(root)
research = research_evidence(root)
sensitivity = diagnostic_evidence(root)
pairs = pairs_evidence(root)
robustness = robustness_evidence(root)
assert all(item is not None for item in (controls, research, sensitivity, pairs, robustness))
gate = feature_gate(root)
instructions = instruction_evidence(root)
released = released_evidence(root)
assert released is not None
print("Verified research runs:", gate["studies"])


Verified research runs: {'research': '9ec008d506b0bc64a717', 'sensitivity': 'bce07fd60bc7543fca49', 'pairs': '537cf2213c813b8ebd4b', 'robustness': 'a09455ec14e9b3d6ab61', 'instructions': '96bf69f42f9063e01a12'}


In [5]:
display(pd.DataFrame([{"Study": name, "Run": run_id} for name, run_id in gate["studies"].items()]))

,Study,Run
0,research,9ec008d506b0bc64a717
1,sensitivity,bce07fd60bc7543fca49
2,pairs,537cf2213c813b8ebd4b
3,robustness,a09455ec14e9b3d6ab61
4,instructions,96bf69f42f9063e01a12


## Review path
Continue to [01 · Validation](01_data_and_validation.ipynb), then [02 · Feature research](02_baseline_and_review.ipynb). [03 · Results](03_saved_results.ipynb) provides the compact decision view. Reading aggregate evidence and recomputing private OOF metrics are distinct operations; the latter is documented in [VALIDATION.md](../docs/VALIDATION.md).